In [ ]:
!pip install tensorflow pandas scikit-learn numpy -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ai_pile_balanced.csv')

print(df.shape)
print(df['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(len(X_train), len(X_test))

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print("Tokenization complete!")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),

    LSTM(64, dropout=0.2, recurrent_dropout=0.2),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.1
)


In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

In [ ]:
import numpy as np

y_pred_prob = model.predict(X_test_pad)

y_pred = (y_pred_prob > 0.5).astype(int)

print("Prediction completed!")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=['Human', 'AI']
))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Human', 'AI']
)

disp.plot()

plt.show()

In [ ]:
model.save('/content/drive/MyDrive/lstm_ai_detector.keras')

print("Model Saved!")

In [ ]:
import pickle

with open('/content/drive/MyDrive/lstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizer Saved!")

In [ ]:
from tensorflow.keras.models import load_model
import pickle

model = load_model('/content/drive/MyDrive/lstm_ai_detector.keras')

with open('/content/drive/MyDrive/lstm_tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print("Model Loaded!")

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 200

def predict_text(text):

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding='post',
        truncating='post'
    )

    prob = model.predict(padded, verbose=0)[0][0]

    if prob > 0.5:
        print(f"🤖 AI Generated ({prob*100:.2f}%)")
    else:
        print(f"👤 Human Written ({(1-prob)*100:.2f}%)")

In [ ]:
while True:

    text = input("\nEnter text (or quit): ")

    if text.lower() == "quit":
        break

    predict_text(text)